In [32]:
import os
import json


In [33]:
data_dir = "./data"

session = "CarpeDiem-26-05-2026"

source_dir = os.path.join(
	data_dir,
	"source",
	session
)

raw_dir = os.path.join(
	data_dir,
	"raw",
	session
)



In [34]:
def load_json(path):
	with open(path, encoding="utf-8") as f:
		return json.load(f)

def save_json(path: str, data):
	if not path.endswith(".json"):
		path += ".json"
	
	with open(path, "w", encoding="utf-8") as f:
		json.dump(data, f, ensure_ascii=False, indent=4)


In [35]:
def normalize_token(token: str):
	if token.startswith("18_"):
		return token[3:]

	return token


In [36]:
for phase in ["Pre", "Post"]:
	phase_source_dir = os.path.join(source_dir, phase)
	phase_raw_dir = os.path.join(raw_dir, phase)
	os.makedirs(phase_raw_dir, exist_ok=True)

	code_file = os.path.join(phase_source_dir, "code.json")
	full_file = os.path.join(phase_source_dir, "full.json")

	raw_code_file = os.path.join(phase_raw_dir, "code.json")
	raw_full_file = os.path.join(phase_raw_dir, "full.json")

	code_data = load_json(code_file)
	full_data = load_json(full_file)

	full_by_token = {
		normalize_token(response["Código de acceso"]): response
		for response in full_data.get("responses", [])
		if response.get("Código de acceso")
	}

	code_result = {}
	full_result = {}

	for code_response in code_data.get("responses", []):
		original_token = code_response.get("token")

		if not token:
			print(f"{phase}: respuesta sin token")
			continue

		token = normalize_token(original_token)

		full_response = full_by_token.get(token)

		if full_response is None:
			print(f"{phase}: token '{token}' no encontrado en full")
			continue

		code_response["token"] = token
		full_response["Código de acceso"] = token

		code_result[token] = code_response
		full_result[token] = full_response

	save_json(raw_code_file, code_result)
	save_json(raw_full_file, full_result)

	print(f"{session}/{phase}: {len(code_result)} respuestas")


CarpeDiem-26-05-2026/Pre: 40 respuestas
CarpeDiem-26-05-2026/Post: 35 respuestas
